# Gold — POE tables and dimension fixes (2026-10-20)

Records the Gold builds run on 2026-10-20 with Span's approval, so they can be reproduced.
Nothing runs until the `build_approved` widget is set to `true`.

## What it builds
| Table | Grain | Why |
|---|---|---|
| `dim_patient` | one row per `subject_id` | Previous table was unreadable (`UC_DEPENDENCY_DOES_NOT_EXIST`); rebuilt |
| `dim_provider` | one row per `provider_id` | Same |
| `dim_careunit` | one row per care unit name in `icustays` and `transfers` | Previous table held only 17 units; 2,378,623 `fact_transfer` rows had no match |
| `dim_date` | one row per shifted day, 2100-01-01 to 2250-12-31 | Previous range ended 2210-12-31; source dates run to 2214 |
| `dim_order_type` | one row per (`order_type`, `order_subtype`) | New, for POE |
| `fact_order` | one row per `poe_id` | New, from `silver.poe` |
| `fact_transfer` | one row per `transfer_id` | Re-keyed: `careunit_key` was `xxhash64(upper(careunit))`, now `xxhash64(careunit)` to match `dim_careunit` and `fact_icu_stay` |

## Key encoding (matches the existing Gold facts)
The existing Gold facts use `xxhash64(CAST(subject_id AS BIGINT))`, `xxhash64(provider_id)` and
`xxhash64(col1, col2)`, **not** the self-delimiting encoding described in `00_Gold_Registry`.
These builds use the facts' encoding so every foreign key joins. A full Gold rewrite must
regenerate dimensions and facts together under one encoding.

## Not built here
`fact_order_detail`: `silver.poe_detail` still fails every read with `UC_DEPENDENCY_DOES_NOT_EXIST`.

## Result on 2026-10-20
49 foreign keys across 12 facts checked against their dimensions: 0 orphan keys.
NULL keys remain only where the source value is NULL (e.g. 308,039 orders with no provider).

In [0]:
dbutils.widgets.text("build_approved", "false")
BUILD_APPROVED = dbutils.widgets.get("build_approved").strip().lower() == "true"

S = "cdr_mimiciv.silver"
G = "cdr_mimiciv.gold"

BUILDS = {}

BUILDS["dim_patient"] = f"""
CREATE OR REPLACE TABLE {G}.dim_patient
COMMENT 'One row per patient (subject_id) with demographics. Date of death is deliberately not carried. patient_key = xxhash64(CAST(subject_id AS BIGINT)), the encoding the existing Gold facts use.'
AS SELECT
    xxhash64(CAST(subject_id AS BIGINT)) AS patient_key,
    subject_id, gender, anchor_age, anchor_year, anchor_year_group,
    age_deidentified_90_plus AS age_deidentified_flag
FROM {S}.patients
WHERE subject_id IS NOT NULL"""

BUILDS["dim_provider"] = f"""
CREATE OR REPLACE TABLE {G}.dim_provider
COMMENT 'Provider identifiers (de-identified). No other provider attributes exist in MIMIC-IV. provider_key = xxhash64(provider_id), the encoding the existing Gold facts use.'
AS SELECT xxhash64(provider_id) AS provider_key, provider_id
FROM {S}.provider
WHERE provider_id IS NOT NULL"""

BUILDS["dim_careunit"] = f"""
CREATE OR REPLACE TABLE {G}.dim_careunit
COMMENT 'Distinct care unit names seen in ICU stays (first/last) and transfers. Name variants (UNKNOWN/Unknown) are kept separate. careunit_key = xxhash64(careunit_name).'
AS SELECT xxhash64(careunit_name) AS careunit_key, careunit_name
FROM (
    SELECT first_careunit AS careunit_name FROM {S}.icustays
    UNION SELECT last_careunit FROM {S}.icustays
    UNION SELECT careunit FROM {S}.transfers
) u
WHERE careunit_name IS NOT NULL"""

BUILDS["dim_date"] = f"""
CREATE OR REPLACE TABLE {G}.dim_date
COMMENT 'Calendar dimension over the SHIFTED MIMIC-IV date range (dates are shifted per patient; no real-calendar meaning). One row per day, 2100-01-01 to 2250-12-31.'
AS SELECT
    CAST(date_format(d, 'yyyyMMdd') AS INT) AS date_key,
    d AS full_date, year(d) AS year, quarter(d) AS quarter, month(d) AS month,
    date_format(d, 'MMMM') AS month_name, day(d) AS day_of_month,
    weekofyear(d) AS week_of_year, dayofweek(d) AS day_of_week, date_format(d, 'EEEE') AS day_name
FROM (SELECT explode(sequence(DATE '2100-01-01', DATE '2250-12-31', INTERVAL 1 DAY)) AS d) t"""

BUILDS["dim_order_type"] = f"""
CREATE OR REPLACE TABLE {G}.dim_order_type
COMMENT 'One row per distinct (order_type, order_subtype) in Silver poe. order_type has 16 documented values (MIMIC-IV hosp poe); order_subtype is best read with order_type and is NULL for most Medications and Lab orders. order_type_key = xxhash64(order_type, order_subtype).'
AS SELECT xxhash64(order_type, order_subtype) AS order_type_key, order_type, order_subtype
FROM (SELECT DISTINCT order_type, order_subtype FROM {S}.poe) t"""

BUILDS["fact_order"] = f"""
CREATE OR REPLACE TABLE {G}.fact_order
COMMENT 'One row per provider order (poe_id, unique; MIMIC-IV hosp poe). Source: silver.poe, no rows filtered, Silver dq_* flags carried. Foreign keys come from LEFT JOINs to gold.dim_patient, gold.dim_order_type and gold.dim_provider (NULL when unmatched). order_date_key = yyyyMMdd of shifted ordertime. is_discontinued = discontinued_by_poe_id IS NOT NULL (our rule).'
AS SELECT
    o.poe_id, o.poe_seq, p.patient_key, o.hadm_id, ot.order_type_key,
    pr.provider_key AS order_provider_key,
    CAST(date_format(o.ordertime, 'yyyyMMdd') AS INT) AS order_date_key,
    o.ordertime, o.transaction_type, o.order_status,
    o.discontinue_of_poe_id, o.discontinued_by_poe_id,
    o.discontinued_by_poe_id IS NOT NULL AS is_discontinued,
    o.dq_hadm_not_in_admissions, o.dq_discontinue_of_poe_id_not_in_poe,
    o.dq_discontinued_by_poe_id_not_in_poe, o.dq_ordertime_before_previous_seq
FROM {S}.poe o
LEFT JOIN {G}.dim_patient p ON o.subject_id = p.subject_id
LEFT JOIN {G}.dim_order_type ot ON o.order_type <=> ot.order_type AND o.order_subtype <=> ot.order_subtype
LEFT JOIN {G}.dim_provider pr ON o.order_provider_id = pr.provider_id"""

# reproduces the 2026-10-20 table exactly (verified column by column against the live table),
# including the lower-case eventtype the earlier build produced ('ed')
BUILDS["fact_transfer"] = f"""
CREATE OR REPLACE TABLE {G}.fact_transfer
COMMENT 'One row per patient transfer/location stay with derived duration and Silver data-quality flags. careunit_key = xxhash64(careunit) to match dim_careunit.'
AS SELECT
    t.transfer_id,
    xxhash64(CAST(t.subject_id AS BIGINT)) AS patient_key,
    t.hadm_id,
    xxhash64(t.careunit) AS careunit_key,
    CAST(date_format(t.intime, 'yyyyMMdd') AS INT) AS in_date_key,
    CAST(date_format(t.outtime, 'yyyyMMdd') AS INT) AS out_date_key,
    lower(t.eventtype) AS eventtype,
    t.intime, t.outtime,
    CASE WHEN t.intime IS NOT NULL AND t.outtime IS NOT NULL
         THEN CAST(unix_micros(t.outtime) - unix_micros(t.intime) AS DOUBLE) / 3600000000.0 END AS transfer_duration_hours,
    t.dq_invalid_transfer_time
FROM {S}.transfers t"""

ORDER = ["dim_patient", "dim_provider", "dim_careunit", "dim_date", "dim_order_type", "fact_order", "fact_transfer"]

In [0]:
if not BUILD_APPROVED:
    raise RuntimeError("Set widget build_approved=true to run these builds.")

for name in ORDER:
    spark.sql(BUILDS[name])
    print("Built:", f"{G}.{name}")

In [0]:
# post-build checks: grain uniqueness and foreign-key matches for the tables built here
checks = {
    "dim_patient": "SELECT COUNT(*) = COUNT(DISTINCT patient_key) FROM {G}.dim_patient",
    "dim_provider": "SELECT COUNT(*) = COUNT(DISTINCT provider_key) FROM {G}.dim_provider",
    "dim_careunit": "SELECT COUNT(*) = COUNT(DISTINCT careunit_key) FROM {G}.dim_careunit",
    "dim_order_type": "SELECT COUNT(*) = COUNT(DISTINCT order_type_key) FROM {G}.dim_order_type",
    "fact_order": "SELECT COUNT(*) = COUNT(DISTINCT poe_id) AND COUNT(*) = (SELECT COUNT(*) FROM {S}.poe) FROM {G}.fact_order",
    "fact_transfer": "SELECT COUNT(*) = COUNT(DISTINCT transfer_id) AND COUNT(*) = (SELECT COUNT(*) FROM {S}.transfers) FROM {G}.fact_transfer",
}
for name, sql in checks.items():
    ok = spark.sql(sql.format(G=G, S=S)).first()[0]
    print(name, "grain OK" if ok else "GRAIN CHECK FAILED")
    assert ok, f"{name}: grain check failed"

FKS = [
    ("fact_order", "patient_key", "dim_patient", "patient_key"),
    ("fact_order", "order_type_key", "dim_order_type", "order_type_key"),
    ("fact_order", "order_provider_key", "dim_provider", "provider_key"),
    ("fact_order", "order_date_key", "dim_date", "date_key"),
    ("fact_transfer", "careunit_key", "dim_careunit", "careunit_key"),
    ("fact_transfer", "patient_key", "dim_patient", "patient_key"),
    ("fact_icu_stay", "first_careunit_key", "dim_careunit", "careunit_key"),
    ("fact_icu_stay", "last_careunit_key", "dim_careunit", "careunit_key"),
    ("fact_admission", "patient_key", "dim_patient", "patient_key"),
    ("fact_prescription", "provider_key", "dim_provider", "provider_key"),
    ("fact_microbiology", "provider_key", "dim_provider", "provider_key"),
    ("fact_lab", "lab_date_key", "dim_date", "date_key"),
    ("fact_icu_observation", "chart_date_key", "dim_date", "date_key"),
]
for f, c, d, k in FKS:
    orphans = spark.sql(f"""
        SELECT COUNT(*) FROM (SELECT DISTINCT {c} FROM {G}.{f} WHERE {c} IS NOT NULL) x
        LEFT ANTI JOIN {G}.{d} y ON x.{c} = y.{k}""").first()[0]
    print(f"{f}.{c} -> {d}: {orphans} orphan keys")
    assert orphans == 0, f"{f}.{c} has keys missing from {d}"

In [0]:
%sql
ALTER TABLE cdr_mimiciv.gold.fact_admission ADD COLUMN (age_at_admission INT);
 
MERGE INTO cdr_mimiciv.gold.fact_admission t
USING (
  SELECT f.hadm_id,
         p.anchor_age + (year(f.admittime) - p.anchor_year) AS age_at_admission
  FROM cdr_mimiciv.gold.fact_admission f
  LEFT JOIN cdr_mimiciv.gold.dim_patient p ON f.patient_key = p.patient_key
) s
ON t.hadm_id = s.hadm_id
WHEN MATCHED THEN UPDATE SET t.age_at_admission = s.age_at_admission;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
546028,546028,0,0
